# EDA — Bronze Layer Analysis
## Exploratory Data Analysis for Silver Layer Design

**Project:** Fornecedor 360 Público em Databricks  
**Purpose:** Profile all Bronze Parquet sources to define Silver transformation rules.  
**Output:** Per-source data quality metrics, sentinel catalog, **actual Parquet column types**,
and a consolidated Silver Rules contract.

## Sources
- `rf_empresas` · `rf_estabelecimentos` · `rf_simples` · `rf_cnaes` · `rf_municipios` · `rf_naturezas` · `rf_paises` · `rf_qualificacoes` · `rf_motivos`
- `compras_contratos` · `pncp_contratos`
- `transp_ceis` · `transp_cnep`

## Sections
1. Setup
2. Data Inventory — file structure + schemas + **actual Parquet types**
3. CNPJ Key Analysis — format reconciliation across sources
4. rf_empresas
5. rf_estabelecimentos
6. rf_simples
7. compras_contratos
8. pncp_contratos
9. transp_ceis + transp_cnep
10. Silver Rules — consolidated transformation contract

## Critical Note on Types
Bronze Parquet files may infer types differently than the raw source suggests.
For example, `data_opcao_simples` in `rf_simples` is BIGINT in Parquet despite
appearing as a date string in the raw CSV. **Always verify types from the Parquet
DESCRIBE, not from the source documentation or raw file inspection.**

## Notes
- Local environment: DuckDB + Parquet (Bronze partitioned as `_year_month=.../data.parquet`)
- Results stored in dicts (`rfe`, `rfest`, `rfs`, `cc`, `pncp_r`, `tc`) for dynamic summaries
- ADR-002: CNPJ always VARCHAR — never INTEGER
- ADR-005: Pseudonymization (HMAC-SHA256) applied at Silver→Gold transition, not here
- ADR-007: Silver filtered to ~190k active CNPJs — this EDA profiles the full Bronze scope

In [ ]:
# =============================================================================
# Section 1 — Setup
# =============================================================================

import sys
from pathlib import Path
from datetime import datetime

# --- Resolve root ------------------------------------------------------------
# Notebook está em: fornecedor360-local/notebooks/eda/eda_bronze.ipynb
# utils/ está em:   fornecedor360-local/utils/
# Logo: root = nb_file.parent.parent.parent
try:
    _nb_file = Path(__vsc_ipynb_file__).resolve()
    _root_candidate = _nb_file.parent.parent.parent  # eda/ → notebooks/ → fornecedor360-local/
except NameError:
    # Fallback: busca pelo marcador utils/paths.py subindo a partir de cwd
    _root_candidate = Path.cwd()
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
        if (candidate / "utils" / "paths.py").exists():
            _root_candidate = candidate
            break

sys.path.insert(0, str(_root_candidate / "utils"))

print(f"Root candidate : {_root_candidate}")
print(f"utils exists   : {(_root_candidate / 'utils' / 'paths.py').exists()}")

from paths        import get_project_root, to_sql_path
from duckdb_utils import get_connection

ROOT       = get_project_root()
BRONZE_DIR = ROOT / "data" / "bronze"
BD         = to_sql_path(BRONZE_DIR)

con = get_connection()  # in-memory — EDA never writes

# Result stores
rfe    = {}   # rf_empresas
rfest  = {}   # rf_estabelecimentos
rfs    = {}   # rf_simples
cc     = {}   # compras_contratos
pncp_r = {}   # pncp_contratos
tc     = {}   # transp_ceis + transp_cnep
cnpj_r = {}   # cross-source CNPJ analysis

def sep(title="", w=68):
    if title:
        print(f"\n{'─'*w}")
        print(f"  {title}")
        print(f"{'─'*w}")
    else:
        print(f"{'─'*w}")

def pct(n, total):
    return 0.0 if total == 0 else round(n / total * 100, 1)

print(f"\nROOT       : {ROOT}")
print(f"BRONZE_DIR : {BRONZE_DIR}")
print(f"DuckDB     : {__import__('duckdb').__version__}")
print(f"Started    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nSetup complete.")

In [ ]:
# =============================================================================
# Section 2 — Data Inventory
# =============================================================================

sep("FILE STRUCTURE — BRONZE LAYER")
print()
inv = {}
for source_dir in sorted(BRONZE_DIR.iterdir()):
    if not source_dir.is_dir():
        continue
    files = list(source_dir.rglob("*.parquet"))
    total_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)
    inv[source_dir.name] = {"files": len(files), "mb": round(total_mb, 1)}
    print(f"  {source_dir.name:<35} {len(files):>3} file(s)   {total_mb:>8,.1f} MB")

print(f"\n  Total Bronze size: {sum(v['mb'] for v in inv.values()):,.1f} MB")

In [ ]:
# =============================================================================
# Section 2b — Schemas + ACTUAL PARQUET TYPES (critical for Silver design)
#
# This is the authoritative source of column types for Silver development.
# Do NOT rely on source documentation or raw file inspection — always use
# the actual Parquet types from DESCRIBE below.
# =============================================================================

sep("SCHEMAS + ACTUAL PARQUET TYPES — ALL SOURCES")

SOURCES = [
    "rf_empresas", "rf_estabelecimentos", "rf_simples",
    "rf_cnaes", "rf_municipios", "rf_naturezas",
    "rf_paises", "rf_qualificacoes", "rf_motivos",
    "compras_contratos", "pncp_contratos",
    "transp_ceis", "transp_cnep",
]
schemas = {}
for src in SOURCES:
    print(f"\n── {src.upper()} ──")
    cols = con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{BD}/{src}/**/*.parquet')"
    ).fetchall()
    schemas[src] = {c[0]: c[1] for c in cols}
    for c in cols:
        if not c[0].startswith("_"):
            flag = " ← CHECK: date stored as BIGINT?" if c[1] == "BIGINT" and "data" in c[0].lower() else ""
            print(f"  {c[0]:<40} {c[1]}{flag}")

In [ ]:
# =============================================================================
# Section 3 — CNPJ Key Analysis
# =============================================================================

sep("CNPJ FORMAT PER SOURCE")

r = con.execute(f"SELECT cnpj_basico, length(cnpj_basico) AS len FROM read_parquet('{BD}/rf_empresas/**/*.parquet') LIMIT 3").fetchall()
cnpj_r['rf_empresas'] = {'field': 'cnpj_basico', 'len': r[0][1], 'example': r[0][0], 'format': 'digits only / ROOT 8'}
print(f"  rf_empresas        field=cnpj_basico           len={r[0][1]}  ex={r[0][0]!r}")

r = con.execute(f"SELECT cnpj_basico||cnpj_ordem||cnpj_dv AS full, length(cnpj_basico||cnpj_ordem||cnpj_dv) AS len FROM read_parquet('{BD}/rf_estabelecimentos/**/*.parquet') LIMIT 3").fetchall()
cnpj_r['rf_estabelecimentos'] = {'field': 'cnpj_basico+cnpj_ordem+cnpj_dv', 'len': r[0][1], 'example': r[0][0], 'format': 'digits only / FULL 14'}
print(f"  rf_estabelecimentos field=basico+ordem+dv       len={r[0][1]}  ex={r[0][0]!r}")

r = con.execute(f"SELECT cnpj_basico, length(cnpj_basico) AS len FROM read_parquet('{BD}/rf_simples/**/*.parquet') LIMIT 3").fetchall()
cnpj_r['rf_simples'] = {'field': 'cnpj_basico', 'len': r[0][1], 'example': r[0][0], 'format': 'digits only / ROOT 8'}
print(f"  rf_simples         field=cnpj_basico           len={r[0][1]}  ex={r[0][0]!r}")

r = con.execute(f"SELECT niFornecedor, length(niFornecedor) AS len FROM read_parquet('{BD}/compras_contratos/**/*.parquet') WHERE length(niFornecedor)=14 LIMIT 3").fetchall()
cnpj_r['compras_contratos'] = {'field': 'niFornecedor', 'len': r[0][1], 'example': r[0][0], 'format': 'digits only / FULL 14'}
print(f"  compras_contratos  field=niFornecedor          len={r[0][1]}  ex={r[0][0]!r}")

r = con.execute(f"SELECT niFornecedor, length(niFornecedor) AS len FROM read_parquet('{BD}/pncp_contratos/**/*.parquet') WHERE length(niFornecedor)=14 LIMIT 3").fetchall()
cnpj_r['pncp_contratos'] = {'field': 'niFornecedor', 'len': r[0][1], 'example': r[0][0], 'format': 'digits only / FULL 14'}
print(f"  pncp_contratos     field=niFornecedor          len={r[0][1]}  ex={r[0][0]!r}")

r = con.execute(f"SELECT pessoa_cnpjFormatado, length(pessoa_cnpjFormatado) AS len FROM read_parquet('{BD}/transp_ceis/**/*.parquet') WHERE pessoa_cnpjFormatado IS NOT NULL AND length(pessoa_cnpjFormatado)=18 LIMIT 3").fetchall()
cnpj_r['transp_ceis'] = {'field': 'pessoa_cnpjFormatado', 'len': r[0][1], 'example': r[0][0], 'format': 'WITH punctuation XX.XXX.XXX/YYYY-ZZ'}
print(f"  transp_ceis        field=pessoa_cnpjFormatado  len={r[0][1]}  ex={r[0][0]!r}")
cnpj_r['transp_cnep'] = cnpj_r['transp_ceis'].copy()
print(f"  transp_cnep        field=pessoa_cnpjFormatado  len=18  (same format)")

sep("CNPJ NORMALIZATION RULES SUMMARY")
print()
rules = [
    ("rf_empresas",         "cnpj_basico",             "8",  "N/A (root only)",            "cnpj_basico directly"),
    ("rf_estabelecimentos", "basico+ordem+dv concat",  "14", "concat directly",            "cnpj_basico directly"),
    ("rf_simples",          "cnpj_basico",             "8",  "N/A (root only)",            "cnpj_basico directly"),
    ("compras_contratos",   "niFornecedor",            "14", "directly (already clean)",   "niFornecedor[:8]"),
    ("pncp_contratos",      "niFornecedor",            "14", "directly (already clean)",   "niFornecedor[:8]"),
    ("transp_ceis/cnep",    "pessoa_cnpjFormatado",    "18", "REGEXP_REPLACE('[^0-9A-Za-z]','','g')", "result[:8]"),
]
print(f"  {'Source':<25} {'Field':<30} {'Len':<5} {'→ cnpj_normalized':<35} {'→ cnpj_basico'}")
print(f"  {'─'*25} {'─'*30} {'─'*5} {'─'*35} {'─'*20}")
for r in rules:
    print(f"  {r[0]:<25} {r[1]:<30} {r[2]:<5} {r[3]:<35} {r[4]}")
print()
print("  NOTE: REGEXP_REPLACE uses '[^0-9A-Za-z]' (ADR-002) — compatible with")
print("  alphanumeric CNPJs launching July 2026 (IN RFB 2.229/2024).")

In [ ]:
# =============================================================================
# Section 4 — rf_empresas
# =============================================================================

sep("rf_empresas — EXPLORATION")
_p = f"{BD}/rf_empresas/**/*.parquet"

r = con.execute(f"SELECT count(*) AS n, count(DISTINCT cnpj_basico) AS d FROM read_parquet('{_p}')").fetchone()
rfe['total'], rfe['duplicatas'] = r[0], r[0] - r[1]
print(f"  Total rows       : {rfe['total']:>15,}")
print(f"  Distinct cnpj    : {r[1]:>15,}")
print(f"  Duplicates       : {rfe['duplicatas']:>15,}  {'OK' if rfe['duplicatas']==0 else 'WARN'}")

sep("Nulls per column")
rfe['nulos'] = {}
cols = ['cnpj_basico','razao_social','natureza_juridica','qualificacao_responsavel',
        'capital_social','porte_empresa','ente_federativo_responsavel']
for c in cols:
    n = con.execute(f"SELECT count(*) FILTER (WHERE {c} IS NULL) FROM read_parquet('{_p}')").fetchone()[0]
    e = con.execute(f"SELECT count(*) FILTER (WHERE {c} = '') FROM read_parquet('{_p}')").fetchone()[0]
    rfe['nulos'][c] = n + e
    flag = 'WARN' if (n+e)/rfe['total'] > 0.01 else 'OK'
    print(f"  {c:<35} null+empty={n+e:>12,}  ({pct(n+e,rfe['total']):>5.1f}%)  {flag}")

sep("capital_social — format sample")
df = con.execute(f"SELECT capital_social, count(*) AS q FROM read_parquet('{_p}') WHERE capital_social IS NOT NULL GROUP BY capital_social ORDER BY q DESC LIMIT 8").fetchall()
for row in df: print(f"  {row[0]!r:<20} → {row[1]:>12,}")
zero = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE capital_social = '0,00'").fetchone()[0]
rfe['capital_zero_pct'] = pct(zero, rfe['total'])
rfe['capital_formato'] = 'BR decimal (comma separator, no thousands)'
print(f"  capital_social=0,00 → {zero:,} rows ({rfe['capital_zero_pct']}%)")

sep("natureza_juridica + qualificacao_responsavel — orphan check")
nat_orf = con.execute(f"SELECT count(*) FROM (SELECT DISTINCT natureza_juridica AS c FROM read_parquet('{_p}')) t LEFT JOIN read_parquet('{BD}/rf_naturezas/**/*.parquet') n ON t.c=n.natureza_codigo WHERE n.natureza_codigo IS NULL").fetchone()[0]
qual_orf = con.execute(f"SELECT count(*) FROM (SELECT DISTINCT qualificacao_responsavel AS c FROM read_parquet('{_p}')) t LEFT JOIN read_parquet('{BD}/rf_qualificacoes/**/*.parquet') q ON t.c=q.qualificacao_codigo WHERE q.qualificacao_codigo IS NULL").fetchone()[0]
qual_orf_rows = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') e LEFT JOIN read_parquet('{BD}/rf_qualificacoes/**/*.parquet') q ON e.qualificacao_responsavel=q.qualificacao_codigo WHERE q.qualificacao_codigo IS NULL").fetchone()[0]
rfe['natureza_orfaos'] = nat_orf
rfe['qualificacao_orfaos'] = qual_orf
rfe['qualificacao_orfao_rows'] = qual_orf_rows
print(f"  natureza_juridica orphans   : {nat_orf}  {'OK' if nat_orf==0 else 'WARN'}")
print(f"  qualificacao orphan codes   : {qual_orf}  orphan rows={qual_orf_rows:,}")

sep("porte_empresa — all values")
df = con.execute(f"SELECT porte_empresa, count(*) AS q FROM read_parquet('{_p}') GROUP BY porte_empresa ORDER BY q DESC").fetchall()
rfe['porte'] = {str(row[0]): row[1] for row in df}
for row in df: print(f"  {str(row[0]):<8} → {row[1]:>12,}  ({pct(row[1],rfe['total'])}%)")

In [ ]:
sep("rf_empresas — SUMMARY & SILVER RULES")
print()
print(f"  GRAIN     : 1 row per cnpj_basico (8-digit root)")
print(f"  TOTAL     : {rfe['total']:,}")
print(f"  DUPLICATES: {rfe['duplicatas']}  {'OK - grain intact' if rfe['duplicatas']==0 else 'WARN - dedup needed'}")
print()
print(f"  SILVER TRANSFORMATION RULES")
print(f"    cnpj_basico       → VARCHAR (ADR-002)")
print(f"    razao_social      → VARCHAR")
print(f"    natureza_juridica → code + LEFT JOIN rf_naturezas for desc")
print(f"    qualificacao_resp → code + LEFT JOIN rf_qualificacoes | 1 orphan code '36' → NULL desc")
print(f"    capital_social    → REPLACE(',','.') → TRY_CAST DECIMAL(15,2)")
print(f"    porte_empresa     → MAP: '01'→'Micro Empresa' | '03'→'EPP' | '05'→'Demais' | NULL→NULL")
print(f"    ente_federativo   → VARCHAR (99.9% null — keep for public entities)")
print(f"    capital_social=0,00 ({rfe['capital_zero_pct']}%) → keep as 0.00, not NULL")
print()
print(f"  FILTER    : none — all {rfe['total']:,} entities preserved in Silver (ADR-007 filter at identidade level)")

In [ ]:
# =============================================================================
# Section 5 — rf_estabelecimentos
# =============================================================================

sep("rf_estabelecimentos — EXPLORATION")
_p = f"{BD}/rf_estabelecimentos/**/*.parquet"

r = con.execute(f"SELECT count(*) AS n, count(DISTINCT cnpj_basico||cnpj_ordem||cnpj_dv) AS d FROM read_parquet('{_p}')").fetchone()
rfest['total'], rfest['duplicatas'] = r[0], r[0] - r[1]
print(f"  Total rows       : {rfest['total']:>15,}")
print(f"  Distinct CNPJ-14 : {r[1]:>15,}")
print(f"  Duplicates       : {rfest['duplicatas']:>15,}  {'OK' if rfest['duplicatas']==0 else 'WARN'}")

# CRITICAL: Verify actual Parquet types for ALL date columns
sep("CRITICAL — Actual Parquet types for date columns")
print("  [These types determine the Silver transformation pattern — do not assume VARCHAR]")
date_cols_estab = ['data_situacao_cadastral', 'data_inicio_atividade', 'data_situacao_especial']
rfest['date_types'] = {}
for col in date_cols_estab:
    dtype = schemas.get('rf_estabelecimentos', {}).get(col, 'UNKNOWN')
    rfest['date_types'][col] = dtype
    pattern = "CASE WHEN val=0 THEN NULL ELSE TRY_STRPTIME(CAST(val AS VARCHAR),'%Y%m%d')::DATE" if dtype == 'BIGINT' \
              else "TRY_STRPTIME(val,'%Y%m%d')::DATE" if dtype == 'VARCHAR' \
              else f"UNKNOWN TYPE: {dtype}"
    print(f"  {col:<35} type={dtype:<10}  Silver pattern: {pattern}")

sep("Date sentinel check")
for col in date_cols_estab:
    dtype = rfest['date_types'].get(col, 'UNKNOWN')
    if dtype == 'BIGINT':
        s = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE {col} = 0").fetchone()[0]
        n = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE {col} IS NULL").fetchone()[0]
        print(f"  {col:<35} sentinel(BIGINT 0)={s:>10,}  null={n:>8,}")
    else:
        s = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE {col} = '00000000'").fetchone()[0]
        n = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE {col} IS NULL").fetchone()[0]
        print(f"  {col:<35} sentinel('00000000')={s:>8,}  null={n:>8,}")
rfest['data_situacao_sentinel'] = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE data_situacao_cadastral = 0").fetchone()[0]

sep("situacao_cadastral — all values")
df = con.execute(f"SELECT situacao_cadastral, count(*) AS q FROM read_parquet('{_p}') GROUP BY situacao_cadastral ORDER BY q DESC").fetchall()
map_sit = {'01':'Nula','02':'Ativa','03':'Suspensa','04':'Inapta','05':'Ativa Não Regular','08':'Baixada','99':'Exclusão lógica'}
rfest['situacao'] = {str(r[0]): r[1] for r in df}
for row in df:
    desc = map_sit.get(str(row[0]), '?')
    print(f"  {str(row[0]):<6} {desc:<25} → {row[1]:>12,}  ({pct(row[1],rfest['total'])}%)")

sep("cnae_fiscal_principal — sentinel 8888888")
s = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE cnae_fiscal_principal = '8888888'").fetchone()[0]
rfest['cnae_sentinel'] = s
print(f"  sentinel '8888888' → {s:,} rows ({pct(s,rfest['total'])}%) — 'not informed'")

sep("Domain code orphan check")
mun_orf = con.execute(f"SELECT count(*) FROM (SELECT DISTINCT municipio c FROM read_parquet('{_p}') WHERE municipio IS NOT NULL) t LEFT JOIN read_parquet('{BD}/rf_municipios/**/*.parquet') m ON t.c=m.municipio_codigo WHERE m.municipio_codigo IS NULL").fetchone()[0]
pais_orf_rows = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') e LEFT JOIN read_parquet('{BD}/rf_paises/**/*.parquet') p ON e.pais=p.pais_codigo WHERE e.pais IS NOT NULL AND p.pais_codigo IS NULL").fetchone()[0]
cnae_orf = con.execute(f"SELECT count(*) FROM (SELECT DISTINCT cnae_fiscal_principal c FROM read_parquet('{_p}') WHERE cnae_fiscal_principal IS NOT NULL AND cnae_fiscal_principal != '8888888') t LEFT JOIN read_parquet('{BD}/rf_cnaes/**/*.parquet') c ON t.c=c.cnae_codigo WHERE c.cnae_codigo IS NULL").fetchone()[0]
motivo_orf = con.execute(f"SELECT count(*) FROM (SELECT DISTINCT motivo_situacao_cadastral AS c FROM read_parquet('{_p}') WHERE motivo_situacao_cadastral IS NOT NULL) t LEFT JOIN read_parquet('{BD}/rf_motivos/**/*.parquet') m ON t.c=m.motivo_codigo WHERE m.motivo_codigo IS NULL").fetchone()[0]
rfest['mun_orfaos'] = mun_orf
rfest['pais_orfao_rows'] = pais_orf_rows
rfest['cnae_orfaos'] = cnae_orf
rfest['motivo_orfao_codes'] = motivo_orf
print(f"  municipio orphans : {mun_orf}  {'OK' if mun_orf==0 else 'WARN'}")
print(f"  pais orphan rows  : {pais_orf_rows:,}  (extinct country codes — expected)")
print(f"  cnae orphans      : {cnae_orf}  {'OK' if cnae_orf==0 else 'WARN'}")
print(f"  motivo orphans    : {motivo_orf}  {'OK' if motivo_orf==0 else 'WARN'}")

In [ ]:
# =============================================================================
# Section 6 — rf_simples
# =============================================================================

sep("rf_simples — EXPLORATION")
_p = f"{BD}/rf_simples/**/*.parquet"

r = con.execute(f"SELECT count(*) AS n, count(DISTINCT cnpj_basico) AS d FROM read_parquet('{_p}')").fetchone()
rfs['total'], rfs['duplicatas'] = r[0], r[0] - r[1]
print(f"  Total rows       : {rfs['total']:>15,}")
print(f"  Distinct cnpj    : {r[1]:>15,}")
print(f"  Duplicates       : {rfs['duplicatas']:>15,}  {'OK' if rfs['duplicatas']==0 else 'WARN'}")

# CRITICAL: Verify actual Parquet types for ALL date columns
sep("CRITICAL — Actual Parquet types for date columns in rf_simples")
print("  [rf_simples has MIXED types — some date columns are BIGINT, others VARCHAR]")
date_cols_simples = ['data_opcao_simples', 'data_exclusao_simples', 'data_opcao_mei', 'data_exclusao_mei']
rfs['date_types'] = {}
for col in date_cols_simples:
    dtype = schemas.get('rf_simples', {}).get(col, 'UNKNOWN')
    rfs['date_types'][col] = dtype
    if dtype == 'BIGINT':
        pattern = "CASE WHEN val=0 THEN NULL ELSE TRY_STRPTIME(CAST(val AS VARCHAR),'%Y%m%d')::DATE"
        sentinel_val = "0 (integer)"
    elif dtype == 'VARCHAR':
        pattern = "CASE WHEN val='00000000' THEN NULL ELSE TRY_STRPTIME(val,'%Y%m%d')::DATE"
        sentinel_val = "'00000000' (string)"
    else:
        pattern = f"UNKNOWN — type={dtype}"
        sentinel_val = "UNKNOWN"
    print(f"  {col:<30} type={dtype:<10}  sentinel={sentinel_val}")
    print(f"    Silver pattern: {pattern}")
    print()

sep("Date sentinel counts per column (using correct type comparison)")
rfs['sentinels'] = {}
for col, dtype in rfs['date_types'].items():
    if dtype == 'BIGINT':
        s = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE {col} = 0").fetchone()[0]
    else:
        s = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE {col} = '00000000'").fetchone()[0]
    n = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE {col} IS NULL").fetchone()[0]
    rfs['sentinels'][col] = s
    print(f"  {col:<30} sentinel={s:>12,}  null={n:>12,}  type={dtype}")

sep("opcao_simples + opcao_mei — domain")
df_s = con.execute(f"SELECT opcao_simples, count(*) AS q FROM read_parquet('{_p}') GROUP BY opcao_simples ORDER BY q DESC").fetchall()
df_m = con.execute(f"SELECT opcao_mei, count(*) AS q FROM read_parquet('{_p}') GROUP BY opcao_mei ORDER BY q DESC").fetchall()
rfs['simples_dist'] = {r[0]: r[1] for r in df_s}
rfs['mei_dist'] = {r[0]: r[1] for r in df_m}
print("  opcao_simples:"); [print(f"    {r[0]!r:<8} → {r[1]:>12,}  ({pct(r[1],rfs['total'])}%)") for r in df_s]
print("  opcao_mei:");    [print(f"    {r[0]!r:<8} → {r[1]:>12,}  ({pct(r[1],rfs['total'])}%)") for r in df_m]

mei_sem_simples = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE opcao_mei='S' AND opcao_simples!='S'").fetchone()[0]
rfs['mei_sem_simples'] = mei_sem_simples
print(f"\n  MEI without Simples: {mei_sem_simples} rows  {'OK' if mei_sem_simples==0 else 'WARN — source anomaly'}")

In [ ]:
sep("rf_simples — SUMMARY & SILVER RULES")
print()
print(f"  GRAIN     : 1 row per cnpj_basico (same grain as rf_empresas)")
print(f"  TOTAL     : {rfs['total']:,}")
print(f"  DUPLICATES: {rfs['duplicatas']}  {'OK' if rfs['duplicatas']==0 else 'WARN'}")
print()
print(f"  ACTUAL PARQUET DATE TYPES (MIXED — critical for Silver):")
for col, dtype in rfs['date_types'].items():
    print(f"    {col:<30} {dtype}")
print()
print(f"  SILVER TRANSFORMATION RULES")
print(f"    cnpj_basico    → VARCHAR (JOIN key)")
print(f"    opcao_simples  → MAP 'S'→TRUE | 'N'→FALSE  BOOLEAN")
print(f"    opcao_mei      → MAP 'S'→TRUE | 'N'→FALSE  BOOLEAN")
for col, dtype in rfs['date_types'].items():
    sentinel_count = rfs['sentinels'].get(col, 0)
    if dtype == 'BIGINT':
        print(f"    {col:<28} → CASE WHEN val=0 THEN NULL")
        print(f"    {'':28}     ELSE TRY_STRPTIME(CAST(val AS VARCHAR),'%Y%m%d')::DATE END")
        print(f"    {'':28}     sentinel(0)={sentinel_count:,}")
    else:
        print(f"    {col:<28} → CASE WHEN val='00000000' THEN NULL")
        print(f"    {'':28}     ELSE TRY_STRPTIME(val,'%Y%m%d')::DATE END")
        print(f"    {'':28}     sentinel('00000000')={sentinel_count:,}")
print()
print(f"  WARNING: Do NOT use safe_date_expr() for BIGINT columns — use hardcoded CASE WHEN val=0 pattern")
print(f"  FILTER  : none")

In [ ]:
# =============================================================================
# Section 7 — compras_contratos  (unchanged from original — already correct)
# =============================================================================

sep("compras_contratos — EXPLORATION")
_p = f"{BD}/compras_contratos/**/*.parquet"

r = con.execute(f"SELECT count(*) AS n, count(DISTINCT numeroContrato) AS nc, count(DISTINCT idCompra) AS ic FROM read_parquet('{_p}')").fetchone()
cc['total'] = r[0]
print(f"  Total rows              : {r[0]:>10,}")
print(f"  Distinct numeroContrato : {r[1]:>10,}  (ratio={r[0]/max(r[1],1):.1f}x — multi-event grain)")
print(f"  Distinct idCompra       : {r[2]:>10,}")

sep("niFornecedor — length distribution")
df = con.execute(f"SELECT length(niFornecedor) AS len, count(*) AS q FROM read_parquet('{_p}') WHERE niFornecedor IS NOT NULL GROUP BY len ORDER BY q DESC LIMIT 10").fetchall()
cc['ni_dist'] = {r[0]: r[1] for r in df}
for row in df:
    tipo = 'CNPJ' if row[0]==14 else 'CPF' if row[0]==11 else 'INVALID/OTHER'
    print(f"  len={row[0]:<5} {tipo:<15} → {row[1]:>10,}  ({pct(row[1],cc['total'])}%)")
cc['ni_cnpj']    = cc['ni_dist'].get(14, 0)
cc['ni_cpf']     = cc['ni_dist'].get(11, 0)
cc['ni_invalid'] = cc['total'] - cc['ni_cnpj'] - cc['ni_cpf']

sep("Date format sample")
r = con.execute(f"SELECT dataVigenciaInicial FROM read_parquet('{_p}') WHERE dataVigenciaInicial IS NOT NULL LIMIT 1").fetchone()
print(f"  dataVigenciaInicial sample: {r[0]!r}  → CAST(LEFT(val,10) AS DATE)")
nulos_fim = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE dataVigenciaFinal IS NULL OR dataVigenciaFinal=''").fetchone()[0]
cc['nulos_vigencia_fim'] = nulos_fim
print(f"  dataVigenciaFinal nulls: {nulos_fim:,} ({pct(nulos_fim,cc['total'])}%) — contracts with no end date")

sep("Value format + negatives")
for col in ['valorGlobal','valorParcela','valorAcumulado']:
    r = con.execute(f"SELECT {col} FROM read_parquet('{_p}') WHERE {col} IS NOT NULL AND {col}!='' LIMIT 1").fetchone()
    neg = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE TRY_CAST({col} AS DOUBLE) < 0").fetchone()[0]
    nul = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE {col} IS NULL OR {col}=''").fetchone()[0]
    print(f"  {col:<20} sample={r[0]!r:<15}  negatives={neg:>5,}  null={nul:>8,}")
cc['valor_negativo_global'] = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE TRY_CAST(valorGlobal AS DOUBLE) < 0").fetchone()[0]

sep("contratoExcluido")
df = con.execute(f"SELECT contratoExcluido, count(*) AS q FROM read_parquet('{_p}') GROUP BY contratoExcluido ORDER BY q DESC").fetchall()
cc['excluido'] = {r[0]: r[1] for r in df}
for row in df: print(f"  {row[0]!r:<10} → {row[1]:>10,}  ({pct(row[1],cc['total'])}%)")

In [ ]:
sep("compras_contratos — SUMMARY & SILVER RULES")
print()
print(f"  GRAIN     : 1 row per contractual event (not 1 per contract)")
print(f"  TOTAL     : {cc['total']:,}")
print()
print(f"  niFornecedor breakdown:")
print(f"    CNPJ (len=14) : {cc['ni_cnpj']:>10,}  ({pct(cc['ni_cnpj'],cc['total'])}%)  MVP scope")
print(f"    CPF  (len=11) : {cc['ni_cpf']:>10,}  ({pct(cc['ni_cpf'],cc['total'])}%)  excluded in Silver")
print(f"    Invalid/other : {cc['ni_invalid']:>10,}  ({pct(cc['ni_invalid'],cc['total'])}%)")
print()
print(f"  SILVER TRANSFORMATION RULES")
print(f"    cnpj_normalized    → niFornecedor (len=14 only) | cnpj_basico = LEFT(val,8)")
print(f"    dataVigenciaInicial→ CAST(LEFT(val,10) AS DATE)  — strip T00:00:00")
print(f"    dataVigenciaFinal  → CAST(LEFT(val,10) AS DATE) | NULL if empty ({cc['nulos_vigencia_fim']:,} rows)")
print(f"    valorGlobal/Parcela→ TRY_CAST(val AS DECIMAL(15,2))  — US decimal")
print(f"                         negatives preserved: {cc['valor_negativo_global']} rows (contractual reductions)")
print(f"    valorAcumulado     → TRY_CAST(val AS DECIMAL(15,2)) | NULL if empty")
print(f"    contratoExcluido   → MAP 'true'→TRUE | 'false'→FALSE")
print(f"    unidadesRequisitantes → VARCHAR (inconsistent format)")
print(f"    _fonte             → literal 'compras_gov'")
print()
print(f"  FILTER    : length(niFornecedor)=14 AND contratoExcluido='false'")

In [ ]:
# =============================================================================
# Section 8 — pncp_contratos  (unchanged from original — already correct)
# =============================================================================

sep("pncp_contratos — EXPLORATION")
_p = f"{BD}/pncp_contratos/**/*.parquet"

r = con.execute(f"SELECT count(*) AS n, count(DISTINCT numeroControlePNCP) AS d FROM read_parquet('{_p}')").fetchone()
pncp_r['total'], pncp_r['duplicatas'] = r[0], r[0] - r[1]
print(f"  Total rows           : {pncp_r['total']:>10,}")
print(f"  Distinct PNCP keys   : {r[1]:>10,}")
print(f"  Duplicates           : {pncp_r['duplicatas']:>10,}  {'OK' if pncp_r['duplicatas']==0 else 'WARN'}")

sep("tipoPessoa x codigoPaisFornecedor")
df = con.execute(f"SELECT tipoPessoa, codigoPaisFornecedor, count(*) AS q FROM read_parquet('{_p}') GROUP BY tipoPessoa, codigoPaisFornecedor ORDER BY q DESC LIMIT 12").fetchall()
for row in df:
    label = 'MVP scope' if row[0]=='PJ' else ''
    print(f"  tipo={row[0]!r:<5} pais={str(row[1])!r:<8} → {row[2]:>10,}  ({pct(row[2],pncp_r['total'])}%)  {label}")
pj_bra = con.execute(f"SELECT count(*) FROM read_parquet('{_p}') WHERE tipoPessoa='PJ' AND (codigoPaisFornecedor='BRA' OR codigoPaisFornecedor IS NULL)").fetchone()[0]
pncp_r['pj_bra'] = pj_bra
print(f"\n  PJ+BRA+NULL_pais combined: {pj_bra:,} ({pct(pj_bra,pncp_r['total'])}%)")

sep("Date format sample")
r = con.execute(f"SELECT dataAssinatura, dataPublicacaoPncp FROM read_parquet('{_p}') WHERE dataAssinatura IS NOT NULL LIMIT 1").fetchone()
print(f"  dataAssinatura     : {r[0]!r}  → CAST(val AS DATE) directly")
print(f"  dataPublicacaoPncp : {r[1]!r}  → CAST(LEFT(val,10) AS DATE)")

sep("orgaoEntidade_esferaId — sphere distribution")
df = con.execute(f"SELECT orgaoEntidade_esferaId, count(*) AS q FROM read_parquet('{_p}') GROUP BY orgaoEntidade_esferaId ORDER BY q DESC").fetchall()
pncp_r['esfera'] = {r[0]: r[1] for r in df}
map_esfera = {'F':'Federal','E':'Estadual','M':'Municipal','D':'Distrital','N':'poderId misuse'}
for row in df:
    desc = map_esfera.get(row[0], '?')
    print(f"  {str(row[0]):<6} {desc:<20} → {row[1]:>10,}  ({pct(row[1],pncp_r['total'])}%)")

In [ ]:
sep("pncp_contratos — SUMMARY & SILVER RULES")
print()
print(f"  GRAIN     : 1 row per numeroControlePNCP (unique key — cleanest grain)")
print(f"  TOTAL     : {pncp_r['total']:,}")
print(f"  DUPLICATES: {pncp_r['duplicatas']}  {'OK' if pncp_r['duplicatas']==0 else 'WARN'}")
print()
print(f"  SILVER TRANSFORMATION RULES")
print(f"    cnpj_normalized  → niFornecedor directly (already clean 14-digit)")
print(f"    cnpj_basico      → LEFT(niFornecedor, 8)")
print(f"    dataAssinatura   → CAST(val AS DATE)  — YYYY-MM-DD, no transform needed")
print(f"    dataVigenciaFim  → CAST(val AS DATE)")
print(f"    dataPublicacaoPncp → CAST(LEFT(val,10) AS DATE)  — strip timestamp")
print(f"    valorInicial/Global → TRY_CAST(val AS DECIMAL(15,2))  — US decimal")
print(f"    orgaoEntidade_esferaId → VARCHAR | flag 'N' → _esfera_valida=FALSE")
print(f"    _fonte           → literal 'pncp'")
print()
print(f"  FILTER    : tipoPessoa='PJ' AND (codigoPaisFornecedor='BRA' OR codigoPaisFornecedor IS NULL)")
print(f"              AND length(niFornecedor)=14  (removes 406 PJ records carrying CPF)")

In [ ]:
# =============================================================================
# Section 9 — transp_ceis + transp_cnep
# =============================================================================

sep("transp_ceis + transp_cnep — EXPLORATION")
_pceis = f"{BD}/transp_ceis/**/*.parquet"
_pcnep = f"{BD}/transp_cnep/**/*.parquet"

for src, _p in [('CEIS', _pceis), ('CNEP', _pcnep)]:
    r = con.execute(f"SELECT count(*) AS n, count(DISTINCT id) AS d FROM read_parquet('{_p}')").fetchone()
    tc[f'{src.lower()}_total'] = r[0]
    tc[f'{src.lower()}_dup'] = r[0] - r[1]
    print(f"  {src:<6} total={r[0]:>8,}  distinct_id={r[1]:>8,}  duplicates={r[0]-r[1]:>5}  {'OK' if r[0]-r[1]==0 else 'WARN'}")

sep("pessoa_tipo — distribution")
for src, _p in [('CEIS', _pceis), ('CNEP', _pcnep)]:
    total = tc[f'{src.lower()}_total']
    print(f"  ── {src}")
    df = con.execute(f"SELECT pessoa_tipo, count(*) AS q FROM read_parquet('{_p}') GROUP BY pessoa_tipo ORDER BY q DESC").fetchall()
    tc[f'{src.lower()}_tipos'] = {r[0]: r[1] for r in df}
    for row in df:
        flag = 'PJ scope' if row[0] in ('Entidades Empresariais Privadas','Entidades Sem Fins Lucrativos') else ''
        print(f"    {str(row[0]):<50} {row[1]:>8,}  ({pct(row[1],total)}%)  {flag}")

pj_ceis = sum(v for k,v in tc['ceis_tipos'].items() if k in ('Entidades Empresariais Privadas','Entidades Sem Fins Lucrativos'))
pj_cnep = sum(v for k,v in tc['cnep_tipos'].items() if k in ('Entidades Empresariais Privadas','Entidades Sem Fins Lucrativos'))
tc['pj_ceis'] = pj_ceis
tc['pj_cnep'] = pj_cnep

sep("Date format + sentinel 'Sem informação'")
date_cols = ['dataInicioSancao','dataFimSancao','dataPublicacaoSancao','dataTransitadoJulgado','dataOrigemInformacao']
tc['sentinels'] = {}
for col in date_cols:
    s = con.execute(f"SELECT count(*) FROM read_parquet('{_pceis}') WHERE {col} = 'Sem informação'").fetchone()[0]
    n = con.execute(f"SELECT count(*) FROM read_parquet('{_pceis}') WHERE {col} IS NULL").fetchone()[0]
    tc['sentinels'][col] = s
    sample = con.execute(f"SELECT {col} FROM read_parquet('{_pceis}') WHERE {col} != 'Sem informação' AND {col} IS NOT NULL LIMIT 1").fetchone()
    sample_val = repr(sample[0]) if sample else 'N/A'
    print(f"  {col:<30} sentinel={s:>6,}  null={n:>5,}  sample={sample_val}")

sep("orgaoSancionador_siglaUf NULL — federal organs")
for src, _p in [('CEIS', _pceis), ('CNEP', _pcnep)]:
    pj_total = tc[f'{src.lower()}_pj'] if f'{src.lower()}_pj' in tc else pj_ceis if src=='CEIS' else pj_cnep
    null_uf = con.execute(f"""
        SELECT count(*) FROM read_parquet('{_p}')
        WHERE pessoa_tipo IN ('Entidades Empresariais Privadas','Entidades Sem Fins Lucrativos')
        AND orgaoSancionador_siglaUf IS NULL
    """).fetchone()[0]
    print(f"  {src}: {null_uf:,} PJ records with NULL orgao UF ({pct(null_uf,pj_ceis if src=='CEIS' else pj_cnep)}%)")
print("  [DECISION] Keep NULL — federal organs act nationally, no UF applies")

sep("valorMulta (CNEP only)")
r = con.execute(f"SELECT count(*) AS tot, count(*) FILTER (WHERE valorMulta='0,00') AS zeros FROM read_parquet('{_pcnep}')").fetchone()
tc['multa_zero'] = r[1]
sample = con.execute(f"SELECT valorMulta, TRY_CAST(REPLACE(REPLACE(valorMulta,'.',''),',','.') AS DECIMAL(15,2)) FROM read_parquet('{_pcnep}') WHERE valorMulta IS NOT NULL AND valorMulta != '0,00' LIMIT 3").fetchall()
print(f"  100% filled — {r[0]} rows total, {r[1]} with '0,00' ({pct(r[1],r[0])}%)")
for row in sample: print(f"    {row[0]!r:<25} → {row[1]}")

In [ ]:
sep("transp_ceis + transp_cnep — SUMMARY & SILVER RULES")
print()
print(f"  GRAIN     : 1 row per sanction (company can have multiple)")
print(f"  CEIS total: {tc['ceis_total']:,}  duplicates={tc['ceis_dup']}  {'OK' if tc['ceis_dup']==0 else 'WARN'}")
print(f"  CNEP total: {tc['cnep_total']:,}  duplicates={tc['cnep_dup']}  {'OK' if tc['cnep_dup']==0 else 'WARN'}")
print()
print(f"  PJ SCOPE AFTER FILTER")
print(f"    CEIS PJ: {tc['pj_ceis']:>8,}  ({pct(tc['pj_ceis'],tc['ceis_total'])}%)")
print(f"    CNEP PJ: {tc['pj_cnep']:>8,}  ({pct(tc['pj_cnep'],tc['cnep_total'])}%)")
print()
print(f"  SILVER TRANSFORMATION RULES — SHARED")
print(f"    pessoa_cnpjFormatado → REGEXP_REPLACE(val,'[^0-9A-Za-z]','','g') → cnpj_normalized")
print(f"    cnpj_basico          → LEFT(cnpj_normalized, 8)")
print(f"    dataInicioSancao     → TRY_STRPTIME(val,'%d/%m/%Y')::DATE  — always filled")
for col in ['dataFimSancao','dataPublicacaoSancao','dataTransitadoJulgado','dataOrigemInformacao']:
    s = tc['sentinels'].get(col, 0)
    print(f"    {col:<25} → CASE WHEN='Sem informação' THEN NULL ELSE TRY_STRPTIME(val,'%d/%m/%Y')::DATE  sentinel={s:,}")
print(f"    orgaoSancionador_siglaUf → keep NULL (federal organs have no UF by design)")
print(f"    _fonte               → 'ceis' or 'cnep'")
print()
print(f"  SILVER TRANSFORMATION RULES — CNEP ONLY")
print(f"    valorMulta           → REPLACE('.','')|REPLACE(',','.') → TRY_CAST DECIMAL(15,2)")
print(f"                           '0,00'→0.00 kept ({tc['multa_zero']:,} rows — valid: no monetary penalty)")
print()
print(f"  FILTER: pessoa_tipo IN ('Entidades Empresariais Privadas','Entidades Sem Fins Lucrativos')")

In [ ]:
# =============================================================================
# Section 10 — Silver Rules — Consolidated Transformation Contract
# =============================================================================

sep("SILVER TRANSFORMATION CONTRACT — ALL SOURCES", w=72)

def rule(field, transformation, note=""):
    note_str = f"  # {note}" if note else ""
    print(f"    {field:<38} {transformation}{note_str}")

def section(title):
    print(f"\n  ┌{'─'*68}┐")
    print(f"  │  {title:<66}│")
    print(f"  └{'─'*68}┘")

section("TABLE: silver_identidade (from rf_empresas + rf_estabelecimentos + rf_simples)")
rule("cnpj_normalized",         "cnpj_basico || cnpj_ordem || cnpj_dv  VARCHAR(14)",    "ADR-002")
rule("cnpj_basico",             "VARCHAR — keep as-is",                                  "JOIN key")
rule("razao_social",            "VARCHAR — from rf_empresas LEFT JOIN")
rule("natureza_juridica_cod",   "VARCHAR — code from rf_empresas")
rule("natureza_juridica_desc",  "LEFT JOIN rf_naturezas",                                "0 orphans")
rule("qualificacao_cod",        "VARCHAR — code from rf_empresas")
rule("qualificacao_desc",       "LEFT JOIN rf_qualificacoes | COALESCE NULL→'Nao encontrada'", "1 orphan code")
rule("capital_social",          "TRY_CAST(REPLACE(',','.') AS DECIMAL(15,2))",           f"{rfe.get('capital_zero_pct',0)}% are 0.00")
rule("porte_empresa",           "MAP '01'→ME | '03'→EPP | '05'→Demais | NULL→NULL")
rule("situacao_cadastral_cod",  "VARCHAR — from rf_estabelecimentos")
rule("situacao_cadastral_desc", "inline CASE MAP (7 values, no JOIN needed)")

# Date rules with actual types
print()
print("    ── Date columns (types from actual Parquet DESCRIBE):")
for col, dtype in rfest.get('date_types', {}).items():
    if dtype == 'BIGINT':
        rule(col, f"CASE WHEN val=0 THEN NULL ELSE TRY_STRPTIME(CAST(val AS VARCHAR),'%Y%m%d')::DATE", f"BIGINT sentinel=0")
    else:
        rule(col, f"TRY_STRPTIME(val,'%Y%m%d')::DATE", f"VARCHAR")
for col, dtype in rfs.get('date_types', {}).items():
    sent = rfs.get('sentinels', {}).get(col, 0)
    if dtype == 'BIGINT':
        rule(col, f"CASE WHEN val=0 THEN NULL ELSE TRY_STRPTIME(CAST(val AS VARCHAR),'%Y%m%d')::DATE", f"BIGINT sentinel=0, count={sent:,}")
    else:
        rule(col, f"CASE WHEN val='00000000' THEN NULL ELSE TRY_STRPTIME(val,'%Y%m%d')::DATE", f"VARCHAR sentinel='00000000', count={sent:,}")

print()
rule("cnae_fiscal_principal_cod", "LEFT JOIN rf_cnaes | '8888888'→NULL",               f"sentinel={rfest.get('cnae_sentinel',0):,}")
rule("cnae_fiscal_secundaria",   "VARCHAR keep (comma-separated)",                       "explode at Gold if needed")
rule("municipio_desc",           "LEFT JOIN rf_municipios",                              "0 orphans")
rule("pais_desc",                "LEFT JOIN rf_paises",                                  f"{rfest.get('pais_orfao_rows',0):,} orphan rows → NULL desc")
rule("motivo_situacao_desc",     "LEFT JOIN rf_motivos",                                f"orphan codes={rfest.get('motivo_orfao_codes',0)}")
rule("opcao_simples",            "MAP 'S'→TRUE | 'N'→FALSE  BOOLEAN")
rule("opcao_mei",                "MAP 'S'→TRUE | 'N'→FALSE  BOOLEAN")
rule("correio_eletronico",       "EXCLUDE (PII)")
rule("telefone_1/2, fax",        "EXCLUDE (PII)")
print("    FILTER: ADR-007 — cnpj_basico IN (SELECT cnpj_basico FROM cnpjs_ativos)")
print("    NOTE:   safe_date_expr() NOT suitable for BIGINT columns — use hardcoded CASE WHEN val=0")

section("TABLE: silver_compras")
rule("cnpj_normalized",          "niFornecedor  VARCHAR(14)",                           "FILTER length=14")
rule("cnpj_basico",              "LEFT(niFornecedor, 8)")
rule("dataVigenciaInicial",      "CAST(LEFT(val,10) AS DATE)",                          "strip T00:00:00")
rule("dataVigenciaFinal",        "CAST(LEFT(val,10) AS DATE) | NULL",                   f"{cc.get('nulos_vigencia_fim',0):,} nulls")
rule("valorGlobal/Parcela",      "TRY_CAST(val AS DECIMAL(15,2))",                      f"negatives={cc.get('valor_negativo_global',0)} — keep")
rule("valorAcumulado",           "TRY_CAST(val AS DECIMAL(15,2)) | NULL")
rule("contratoExcluido",         "MAP 'true'→TRUE | 'false'→FALSE")
rule("unidadesRequisitantes",    "VARCHAR keep (inconsistent format)")
rule("_fonte",                   "literal 'compras_gov'")
print("    FILTER: length(niFornecedor)=14 AND contratoExcluido='false'")

section("TABLE: silver_pncp")
rule("cnpj_normalized",          "niFornecedor  VARCHAR(14)",                           "already clean")
rule("cnpj_basico",              "LEFT(niFornecedor, 8)")
rule("dataAssinatura",           "CAST(val AS DATE)",                                   "YYYY-MM-DD")
rule("dataPublicacaoPncp",       "CAST(LEFT(val,10) AS DATE)",                          "strip timestamp")
rule("valorInicial/Global",      "TRY_CAST(val AS DECIMAL(15,2))",                      "US decimal")
rule("orgaoEntidade_esferaId",   "VARCHAR | 'N'→_esfera_valida=FALSE")
rule("_fonte",                   "literal 'pncp'")
print("    FILTER: tipoPessoa='PJ' AND (codigoPaisFornecedor='BRA' OR IS NULL) AND length(niFornecedor)=14")

section("TABLE: silver_ceis")
rule("cnpj_normalized",          "REGEXP_REPLACE(cnpjFormatado,'[^0-9A-Za-z]','','g')", "ADR-002 compatible")
rule("cnpj_basico",              "LEFT(cnpj_normalized, 8)")
rule("dataInicioSancao",         "TRY_STRPTIME(val,'%d/%m/%Y')::DATE",                  "always filled")
for col in ['dataFimSancao','dataPublicacaoSancao','dataTransitadoJulgado','dataOrigemInformacao']:
    s = tc.get('sentinels', {}).get(col, 0)
    rule(col, f"WHEN='Sem informação' THEN NULL ELSE TRY_STRPTIME(val,'%d/%m/%Y')::DATE", f"sentinel={s:,}")
rule("orgaoSancionador_siglaUf", "keep NULL",                                            "federal organs have no UF by design")
rule("_fonte",                   "literal 'ceis'")
print("    FILTER: pessoa_tipo IN ('Entidades Empresariais Privadas','Entidades Sem Fins Lucrativos')")

section("TABLE: silver_cnep  (all CEIS rules + valorMulta)")
rule("valorMulta",               "REPLACE('.','')|REPLACE(',','.')|TRY_CAST DECIMAL(15,2)", f"'0,00'→0.00 kept ({tc.get('multa_zero',0)} rows)")
rule("_fonte",                   "literal 'cnep'")
print("    FILTER: same as CEIS")

sep("CROSS-SOURCE JOIN KEY STRATEGY", w=72)
print()
print("  Full CNPJ  (14-digit) → primary join key for contracts and sanctions")
print("  Root CNPJ  ( 8-digit) → join key to rf_empresas (company-level data)")
print("  Rule: cnpj_basico = LEFT(cnpj_normalized, 8)  — applies to all tables")
print()
print("  ADR-007 — Silver filter:")
print("    cnpj_basico IN (SELECT DISTINCT cnpj_basico FROM silver_ceis")
print("                    UNION ALL silver_cnep UNION ALL silver_compras UNION ALL silver_pncp)")
print("    Reduces silver_identidade from 70M to ~810k rows")
print()
print("  ADR-005 — Gold pseudonymization:")
print("    cnpj_token = HMAC-SHA256(cnpj_normalized, salt)  — replaces raw CNPJ in Gold/Serving")
print("    salt stored in Databricks Secret Scope 'fornecedor360-secrets' key 'cnpj-salt'")
print()
print("  CRITICAL WARNING — Type-aware Silver patterns:")
print("    BIGINT date columns: CASE WHEN val=0 THEN NULL ELSE TRY_STRPTIME(CAST(val AS VARCHAR),'%Y%m%d')::DATE")
print("    VARCHAR date columns: CASE WHEN val='00000000' THEN NULL ELSE TRY_STRPTIME(val,'%Y%m%d')::DATE")
print("    safe_date_expr() in duckdb_utils is for VARCHAR columns ONLY")
print("    Verify type with DESCRIBE before writing any Silver transformation")